In [0]:
base_path = "/Volumes/workspace/default/mandi_project_data/"

files = dbutils.fs.ls(base_path)

for file in files:
    print(file.name, "|", file.size, "bytes")

crop_arrivals.csv | 3543552 bytes
crop_arrivals_fixed.csv | 2467377 bytes
crop_arrivals_wheat_2025.json | 503540 bytes
mandi_daily_2025.parquet | 71858358 bytes
weather_tamil_nadu_2025.csv | 140424 bytes


In [0]:
base_path = "/Volumes/workspace/default/mandi_project_data/"

# 1. Mandi price data
mandi_df = spark.read.parquet(
    base_path + "mandi_daily_2025.parquet"
)

# 2. Wheat arrival data
arrivals_df = (
    spark.read
    .option("multiline", "true")
    .json(base_path + "crop_arrivals_wheat_2025.json")
)

# 3. Weather data
weather_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(base_path + "weather_tamil_nadu_2025.csv")
)

print("All three datasets loaded successfully!")

All three datasets loaded successfully!


In [0]:
print("========== MANDI DATA ==========")
mandi_df.printSchema()
print("Total records:", mandi_df.count())

print("\n========== CROP ARRIVALS DATA ==========")
arrivals_df.printSchema()
print("Total records:", arrivals_df.count())

print("\n========== WEATHER DATA ==========")
weather_df.printSchema()
print("Total records:", weather_df.count())

========== MANDI DATA ==========
root
 |-- State: string (nullable = true)
 |-- District: string (nullable = true)
 |-- Market: string (nullable = true)
 |-- Commodity: string (nullable = true)
 |-- Variety: string (nullable = true)
 |-- Grade: string (nullable = true)
 |-- Arrival_Date: date (nullable = true)
 |-- Min_Price: double (nullable = true)
 |-- Max_Price: double (nullable = true)
 |-- Modal_Price: double (nullable = true)
 |-- Commodity_Code: long (nullable = true)

Total records: 5827604

========== CROP ARRIVALS DATA ==========
root
 |-- cmdty: string (nullable = true)
 |-- district: string (nullable = true)
 |-- district_id: long (nullable = true)
 |-- qty: double (nullable = true)
 |-- state: string (nullable = true)
 |-- state_id: long (nullable = true)
 |-- t: string (nullable = true)

Total records: 2370

========== WEATHER DATA ==========
root
 |-- time: date (nullable = true)
 |-- temperature_2m_mean: double (nullable = true)
 |-- temperature_2m_max: double (nullabl

In [0]:
print("========== MANDI DATA ==========")
mandi_df.show(5, truncate=False)

print("========== CROP ARRIVALS DATA ==========")
arrivals_df.show(5, truncate=False)

print("========== WEATHER DATA ==========")
weather_df.show(5, truncate=False)

========== MANDI DATA ==========
+----------+----------+-----------------------------+---------+---------+-----+------------+---------+---------+-----------+--------------+
|State     |District  |Market                       |Commodity|Variety  |Grade|Arrival_Date|Min_Price|Max_Price|Modal_Price|Commodity_Code|
+----------+----------+-----------------------------+---------+---------+-----+------------+---------+---------+-----------+--------------+
|Tamil Nadu|Theni     |Theni (Uzhavar Sandhai ) APMC|Chow Chow|Chow Chow|Local|2025-12-30  |2400.0   |2400.0   |2400.0     |167           |
|Punjab    |Jalandhar |Adampur APMC                 |Potato   |Local    |FAQ  |2025-12-31  |200.0    |300.0    |300.0      |24            |
|Tamil Nadu|Coimbatore|Vadavalli (Uzhavar Sandhai ) |Apple    |American |Local|2025-01-03  |18000.0  |20000.0  |20000.0    |17            |
|Telangana |Warangal  |Warangal                     |Tomato   |Tomato   |FAQ  |2025-01-08  |1000.0   |1500.0   |1250.0     |78 

In [0]:
from pyspark.sql import functions as F

def inspect_data(df, name):
    print(f"\n========== {name} ==========")

    # Missing values
    print("Missing values:")
    df.select([
        F.count(
            F.when(F.col(c).isNull(), c)
        ).alias(c)
        for c in df.columns
    ]).show(truncate=False)

    # Duplicate records
    print("Duplicate records:", 
          df.count() - df.dropDuplicates().count())


inspect_data(mandi_df, "MANDI DATA")
inspect_data(arrivals_df, "CROP ARRIVALS DATA")
inspect_data(weather_df, "WEATHER DATA")


========== MANDI DATA ==========
Missing values:
+-----+--------+------+---------+-------+-----+------------+---------+---------+-----------+--------------+
|State|District|Market|Commodity|Variety|Grade|Arrival_Date|Min_Price|Max_Price|Modal_Price|Commodity_Code|
+-----+--------+------+---------+-------+-----+------------+---------+---------+-----------+--------------+
|0    |0       |0     |0        |0      |0    |0           |4418     |4718     |0          |0             |
+-----+--------+------+---------+-------+-----+------------+---------+---------+-----------+--------------+

Duplicate records: 0

========== CROP ARRIVALS DATA ==========
Missing values:
+-----+--------+-----------+---+-----+--------+---+
|cmdty|district|district_id|qty|state|state_id|t  |
+-----+--------+-----------+---+-----+--------+---+
|0    |0       |0          |0  |0    |0       |0  |
+-----+--------+-----------+---+-----+--------+---+

Duplicate records: 0

========== WEATHER DATA ==========
Missing valu

In [0]:
print("========== MANDI COVERAGE ==========")

mandi_df.select(
    F.min("Arrival_Date").alias("Start Date"),
    F.max("Arrival_Date").alias("End Date")
).show()

print("States:", mandi_df.select("State").distinct().count())
print("Districts:", mandi_df.select("District").distinct().count())
print("Commodities:", mandi_df.select("Commodity").distinct().count())


print("\n========== ARRIVALS COVERAGE ==========")

arrivals_df.select(
    F.min("t").alias("Start Month"),
    F.max("t").alias("End Month")
).show()

print("States:", arrivals_df.select("state").distinct().count())
print("Districts:", arrivals_df.select("district").distinct().count())
print("Commodities:", arrivals_df.select("cmdty").distinct().count())


print("\n========== WEATHER COVERAGE ==========")

weather_df.select(
    F.min("time").alias("Start Date"),
    F.max("time").alias("End Date")
).show()

print("States:", weather_df.select("State").distinct().count())
print("Districts:", weather_df.select("District").distinct().count())

========== MANDI COVERAGE ==========
+----------+----------+
|Start Date|  End Date|
+----------+----------+
|2025-01-01|2025-12-31|
+----------+----------+

States: 30
Districts: 572
Commodities: 338

========== ARRIVALS COVERAGE ==========
+-----------+---------+
|Start Month|End Month|
+-----------+---------+
|    2025-01|  2025-10|
+-----------+---------+

States: 14
Districts: 288
Commodities: 1

========== WEATHER COVERAGE ==========
+----------+----------+
|Start Date|  End Date|
+----------+----------+
|2025-01-01|2025-12-31|
+----------+----------+

States: 1
Districts: 5


In [0]:
from pyspark.sql import functions as F

# 1. Mandi data
mandi_clean = (
    mandi_df
    .withColumn("State", F.initcap(F.trim("State")))
    .withColumn("District", F.initcap(F.trim("District")))
    .withColumn("Commodity", F.initcap(F.trim("Commodity")))
    .withColumn("Month", F.date_format("Arrival_Date", "yyyy-MM"))
)

# 2. Crop arrivals data
arrivals_clean = (
    arrivals_df
    .withColumn("state", F.initcap(F.trim("state")))
    .withColumn("district", F.initcap(F.trim("district")))
    .withColumn("cmdty", F.initcap(F.trim("cmdty")))
    .withColumnRenamed("state", "State")
    .withColumnRenamed("district", "District")
    .withColumnRenamed("cmdty", "Commodity")
    .withColumnRenamed("qty", "Arrival_Quantity")
    .withColumnRenamed("t", "Month")
)

# 3. Weather data
weather_clean = (
    weather_df
    .withColumn("State", F.initcap(F.trim("State")))
    .withColumn("District", F.initcap(F.trim("District")))
    .withColumn("Month", F.date_format("time", "yyyy-MM"))
)

print("All three datasets standardized successfully!")

All three datasets standardized successfully!


In [0]:
# 1. Mandi monthly aggregation (Wheat only)
mandi_monthly = (
    mandi_clean
    .filter(F.col("Commodity") == "Wheat")
    .groupBy("State", "District", "Month")
    .agg(
        F.avg("Min_Price").alias("Average_Min_Price"),
        F.avg("Max_Price").alias("Average_Max_Price"),
        F.avg("Modal_Price").alias("Average_Modal_Price"),
        F.count("*").alias("Price_Record_Count")
    )
)

# 2. Crop arrivals monthly aggregation
arrivals_monthly = (
    arrivals_clean
    .groupBy("State", "District", "Month")
    .agg(
        F.sum("Arrival_Quantity").alias("Total_Arrival_Quantity")
    )
)

# 3. Weather monthly aggregation
weather_monthly = (
    weather_clean
    .groupBy("State", "District", "Month")
    .agg(
        F.avg("temperature_2m_mean").alias("Average_Temperature"),
        F.avg("temperature_2m_max").alias("Average_Max_Temperature"),
        F.avg("temperature_2m_min").alias("Average_Min_Temperature"),
        F.sum("precipitation_sum").alias("Total_Precipitation"),
        F.avg("relative_humidity_2m_mean").alias("Average_Humidity"),
        F.avg("wind_speed_10m_max").alias("Average_Wind_Speed")
    )
)

print("Monthly aggregation completed!")

Monthly aggregation completed!


In [0]:
print("Mandi monthly records:", mandi_monthly.count())
print("Arrivals monthly records:", arrivals_monthly.count())
print("Weather monthly records:", weather_monthly.count())

Mandi monthly records: 2995
Arrivals monthly records: 2370
Weather monthly records: 60


In [0]:
mandi_keys = mandi_monthly.select("State", "District").distinct()
arrivals_keys = arrivals_monthly.select("State", "District").distinct()
weather_keys = weather_monthly.select("State", "District").distinct()

print("Mandi + Arrivals district matches:",
      mandi_keys.join(arrivals_keys, ["State", "District"], "inner").count())

print("Mandi + Weather district matches:",
      mandi_keys.join(weather_keys, ["State", "District"], "inner").count())

print("Arrivals + Weather district matches:",
      arrivals_keys.join(weather_keys, ["State", "District"], "inner").count())

Mandi + Arrivals district matches: 199
Mandi + Weather district matches: 0
Arrivals + Weather district matches: 0


In [0]:
# Join mandi and arrivals
combined_df = (
    mandi_monthly
    .join(
        arrivals_monthly,
        on=["State", "District", "Month"],
        how="full_outer"
    )
)

# Join weather
combined_df = (
    combined_df
    .join(
        weather_monthly,
        on=["State", "District", "Month"],
        how="full_outer"
    )
)

# Add data availability flags
combined_df = (
    combined_df
    .withColumn(
        "Mandi_Present",
        F.col("Price_Record_Count").isNotNull()
    )
    .withColumn(
        "Arrivals_Present",
        F.col("Total_Arrival_Quantity").isNotNull()
    )
    .withColumn(
        "Weather_Present",
        F.col("Average_Temperature").isNotNull()
    )
)

print("Integration completed!")
print("Total combined records:", combined_df.count())


Integration completed!
Total combined records: 3798


In [0]:
print("========== INTEGRATION VALIDATION ==========")

print("Total records:", combined_df.count())

print("\nData availability:")
combined_df.groupBy(
    "Mandi_Present",
    "Arrivals_Present",
    "Weather_Present"
).count().orderBy(
    "Mandi_Present",
    "Arrivals_Present",
    "Weather_Present"
).show(truncate=False)

print("Duplicate records by State, District and Month:",
      combined_df.groupBy(
          "State", "District", "Month"
      ).count().filter(
          F.col("count") > 1
      ).count())

print("\nSample integrated records:")
combined_df.show(10, truncate=False)


========== INTEGRATION VALIDATION ==========
Total records: 3798

Data availability:
+-------------+----------------+---------------+-----+
|Mandi_Present|Arrivals_Present|Weather_Present|count|
+-------------+----------------+---------------+-----+
|false        |false           |true           |60   |
|false        |true            |false          |743  |
|true         |false           |false          |1368 |
|true         |true            |false          |1627 |
+-------------+----------------+---------------+-----+

Duplicate records by State, District and Month: 0

Sample integrated records:
+--------------+-------------+-------+------------------+------------------+-------------------+------------------+----------------------+-------------------+-----------------------+-----------------------+-------------------+----------------+------------------+-------------+----------------+---------------+
|State         |District     |Month  |Average_Min_Price |Average_Max_Price |Average_Mo

In [0]:
combined_df = combined_df.withColumn(
    "Match_Type",
    F.when(
        F.col("Mandi_Present") & F.col("Arrivals_Present"),
        "Mandi + Arrivals"
    )
    .when(
        F.col("Mandi_Present"),
        "Mandi Only"
    )
    .when(
        F.col("Arrivals_Present"),
        "Arrivals Only"
    )
    .when(
        F.col("Weather_Present"),
        "Weather Only"
    )
    .otherwise("No Data")
)

combined_df.groupBy("Match_Type").count().show()

+----------------+-----+
|      Match_Type|count|
+----------------+-----+
|    Weather Only|   60|
|      Mandi Only| 1368|
|   Arrivals Only|  743|
|Mandi + Arrivals| 1627|
+----------------+-----+



In [0]:
output_path = base_path + "processed/"

dbutils.fs.mkdirs(output_path)

combined_df.write \
    .mode("overwrite") \
    .parquet(output_path + "monthly_combined_intelligence")

print("Integrated dataset saved successfully!")

Integrated dataset saved successfully!


In [0]:
saved_df = spark.read.parquet(
    output_path + "monthly_combined_intelligence"
)

print("Saved records:", saved_df.count())
saved_df.printSchema()


Saved records: 3798
root
 |-- State: string (nullable = true)
 |-- District: string (nullable = true)
 |-- Month: string (nullable = true)
 |-- Average_Min_Price: double (nullable = true)
 |-- Average_Max_Price: double (nullable = true)
 |-- Average_Modal_Price: double (nullable = true)
 |-- Price_Record_Count: long (nullable = true)
 |-- Total_Arrival_Quantity: double (nullable = true)
 |-- Average_Temperature: double (nullable = true)
 |-- Average_Max_Temperature: double (nullable = true)
 |-- Average_Min_Temperature: double (nullable = true)
 |-- Total_Precipitation: double (nullable = true)
 |-- Average_Humidity: double (nullable = true)
 |-- Average_Wind_Speed: double (nullable = true)
 |-- Mandi_Present: boolean (nullable = true)
 |-- Arrivals_Present: boolean (nullable = true)
 |-- Weather_Present: boolean (nullable = true)
 |-- Match_Type: string (nullable = true)



In [0]:
# Save monthly mandi data
mandi_monthly.write.mode("overwrite").parquet(
    output_path + "monthly_mandi_prices"
)

# Save monthly arrivals data
arrivals_monthly.write.mode("overwrite").parquet(
    output_path + "monthly_wheat_arrivals"
)

# Save monthly weather data
weather_monthly.write.mode("overwrite").parquet(
    output_path + "monthly_weather"
)

print("All three aggregated datasets saved successfully!")

All three aggregated datasets saved successfully!


In [0]:
for name in [
    "monthly_mandi_prices",
    "monthly_wheat_arrivals",
    "monthly_weather"
]:
    df = spark.read.parquet(output_path + name)
    print(name, ":", df.count(), "records")

monthly_mandi_prices : 2995 records
monthly_wheat_arrivals : 2370 records
monthly_weather : 60 records


In [0]:
datasets = {
    "Mandi": "monthly_mandi_prices",
    "Arrivals": "monthly_wheat_arrivals",
    "Weather": "monthly_weather",
    "Combined": "monthly_combined_intelligence"
}

for name, folder in datasets.items():
    df = spark.read.parquet(output_path + folder)
    print(f"{name}: {df.count()} records")

print("\nDuplicate combined keys:")

final_df = spark.read.parquet(
    output_path + "monthly_combined_intelligence"
)

final_df.groupBy("State", "District", "Month") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

print("Match type distribution:")
final_df.groupBy("Match_Type").count().show()

Mandi: 2995 records
Arrivals: 2370 records
Weather: 60 records
Combined: 3798 records

Duplicate combined keys:
+-----+--------+-----+-----+
|State|District|Month|count|
+-----+--------+-----+-----+
+-----+--------+-----+-----+

Match type distribution:
+----------------+-----+
|      Match_Type|count|
+----------------+-----+
|    Weather Only|   60|
|   Arrivals Only|  743|
|      Mandi Only| 1368|
|Mandi + Arrivals| 1627|
+----------------+-----+

